# 01 - Channel diagnostics on `mart_weekly_mmm`

Goal: decide which media channels to keep, drop or merge **before** fitting the Bayesian MMM.

Steps:
1. Load the weekly mart from BigQuery and sanity check it
2. Find sparse channels (too few active weeks to estimate)
3. Look at pairwise correlation, raw and after removing trend and seasonality
4. Compute VIF (multicollinearity)
5. Test candidate merges and compare

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import statsmodels.api as sm
from matplotlib.colors import LinearSegmentedColormap
from statsmodels.stats.outliers_influence import variance_inflation_factor

PROJECT = "bayesian-mmm-ecommerce"
DATASET = "bayesian_mmm_ecommerce"
TABLE = "mart_weekly_mmm"

CHANNELS = ["google_paid_search", "google_shopping", "google_pmax", "google_display",
            "google_video", "meta_facebook", "meta_instagram", "meta_other", "tiktok"]
CONTROLS = ["black_friday", "christmas"]
TARGET = "first_purchases"

BLUE, ORANGE, INK, GRID = "#2a78d6", "#eb6834", "#52514e", "#e3e2de"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.edgecolor": GRID, "axes.labelcolor": INK, "xtick.color": INK,
                     "ytick.color": INK, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
                     "axes.titlesize": 10, "axes.titleweight": "regular", "figure.dpi": 110})

## 1. Load the weekly mart

In [ ]:
from google.cloud import bigquery

client = bigquery.Client(project=PROJECT)
w = client.query(f"SELECT * FROM {DATASET}.{TABLE} ORDER BY week_start").to_dataframe()

w["week_start"] = pd.to_datetime(w["week_start"])
w = w.sort_values("week_start").reset_index(drop=True)
print(w.shape)
print("range:", w.week_start.min().date(), "to", w.week_start.max().date())
w.head()

In [ ]:
# integrity checks
gaps = w.week_start.diff().dt.days.dropna()
print("weeks that are not exactly 7 days after the previous:", int((gaps != 7).sum()))
print("nulls:", int(w[CHANNELS + [TARGET] + CONTROLS].isna().sum().sum()))
print("negative spend values:", int((w[CHANNELS] < 0).sum().sum()))
w[[TARGET] + CHANNELS].describe().T.round(0)

## 2. Sparse channels

A channel that spent in only a handful of weeks gives the model almost nothing to learn its curve from.
Rule of thumb used here: keep a channel as a standalone candidate if it was active in at least half of the weeks.

In [ ]:
active_share = (w[CHANNELS] > 0).mean()
total = w[CHANNELS].sum()
summary = pd.DataFrame({"active_weeks": (w[CHANNELS] > 0).sum(),
                        "active_share": active_share.round(2),
                        "total_spend": total.round(0),
                        "spend_share": (total / total.sum()).round(3)}).sort_values("total_spend", ascending=False)
summary

In [ ]:
ACTIVE = [ch for ch in CHANNELS if active_share[ch] >= 0.5]
DROPPED = [ch for ch in CHANNELS if ch not in ACTIVE]
print("kept as candidates:", ACTIVE)
print("sparse (review before using):", DROPPED)

In [ ]:
n = len(ACTIVE) + 1
cols = 3
rows = int(np.ceil(n / cols))
fig, axes = plt.subplots(rows, cols, figsize=(11, 2.6 * rows), sharex=True)
for ax, name in zip(axes.ravel(), [TARGET] + ACTIVE):
    ax.plot(w.week_start, w[name], color=BLUE, lw=1.4)
    ax.set_title(name)
    ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 7]))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %y"))
    ax.tick_params(axis="x", labelbottom=True, labelsize=8)
    for lab in ax.get_xticklabels():
        lab.set_rotation(45); lab.set_ha("right")
for ax in axes.ravel()[n:]:
    ax.set_visible(False)
fig.tight_layout()
plt.show()

## 3. Correlation

Two views. The raw view is dominated by shared seasonality (every channel spends more around Black Friday).
The second view removes trend, holiday flags and yearly seasonality first, so it shows whether channels move together **beyond** the calendar.

In [ ]:
DIVERGING = LinearSegmentedColormap.from_list("div", [BLUE, "#f4f3f0", ORANGE])

def heatmap(corr, title):
    fig, ax = plt.subplots(figsize=(0.75 * len(corr) + 3, 0.65 * len(corr) + 2))
    im = ax.imshow(corr.values, cmap=DIVERGING, vmin=-1, vmax=1)
    ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.index)
    ax.grid(False)
    for i in range(len(corr)):
        for j in range(len(corr)):
            ax.text(j, i, f"{corr.values[i, j]:.2f}", ha="center", va="center", fontsize=8, color="#0b0b0b")
    ax.set_title(title)
    fig.colorbar(im, ax=ax, shrink=0.8)
    fig.tight_layout(); plt.show()

heatmap(w[ACTIVE].corr(), "Raw weekly spend correlation")

In [ ]:
t = np.arange(len(w))
doy = w.week_start.dt.dayofyear.values
Z = pd.DataFrame({"const": 1.0, "t": t,
                  "black_friday": w.black_friday, "christmas": w.christmas,
                  "s1": np.sin(2*np.pi*doy/365.25), "c1": np.cos(2*np.pi*doy/365.25),
                  "s2": np.sin(4*np.pi*doy/365.25), "c2": np.cos(4*np.pi*doy/365.25)})
resid = pd.DataFrame({ch: sm.OLS(w[ch].values, Z).fit().resid for ch in ACTIVE})
heatmap(resid.corr(), "Correlation after removing trend, holidays and seasonality")

In [ ]:
def top_pairs(corr, k=8):
    s = corr.where(np.triu(np.ones(corr.shape), 1).astype(bool)).stack()
    return s.reindex(s.abs().sort_values(ascending=False).index).head(k).round(2)

print("Top raw pairs"); print(top_pairs(w[ACTIVE].corr()))
print("\nTop pairs after removing calendar effects"); print(top_pairs(resid.corr()))

## 4. VIF

VIF says how well each channel is predicted by the **other** channels (and controls).
Rough guide: above 5 deserves attention, above 10 is a problem.
Computed once on the channels alone and once with the calendar controls included.

In [ ]:
def vif_table(X, only=None):
    Xc = sm.add_constant(X)
    v = pd.Series([variance_inflation_factor(Xc.values, i + 1) for i in range(X.shape[1])], index=X.columns)
    v = v[only] if only is not None else v
    return v.round(1).sort_values(ascending=False)

vif_raw = vif_table(w[ACTIVE])
vif_ctrl = vif_table(pd.concat([w[ACTIVE], w[CONTROLS], Z[["t", "s1", "c1", "s2", "c2"]]], axis=1), only=ACTIVE)
pd.DataFrame({"VIF channels only": vif_raw, "VIF with controls": vif_ctrl}).sort_values("VIF channels only", ascending=False)

## 5. Test candidate merges

Groups are decided by business logic first (same platform, same mechanism), then checked against the numbers above.
Each scenario below is a proposal. Edit or add scenarios to try other combinations. Channels not listed stay on their own.

Two things to look at in the earlier output before choosing:
- `google_shopping` spend falls to near zero while `google_pmax` ramps up, and the two are negatively correlated. A common reason is that Shopping campaigns were moved into Performance Max (check with whoever ran the account). If so, they behave as one product over time.
- `meta_other` is tiny (about 1,000 dollars in total), so it carries no signal on its own.

In [ ]:
SCENARIOS = {
    "A: search+shopping, meta combined": {
        "google_search": ["google_paid_search", "google_shopping"],
        "meta": ["meta_facebook", "meta_instagram", "meta_other"],
    },
    "B: shopping+pmax, meta combined": {
        "google_shopping_pmax": ["google_shopping", "google_pmax"],
        "meta": ["meta_facebook", "meta_instagram", "meta_other"],
    },
    "C: only fold meta_other into facebook": {
        "meta_facebook": ["meta_facebook", "meta_other"],
    },
}

def build_merged(w, groups, active):
    out, used = pd.DataFrame(index=w.index), set()
    for name, members in groups.items():
        members = [m for m in members if m in active]
        if members:
            out[name] = w[members].sum(axis=1); used.update(members)
    for ch in active:
        if ch not in used:
            out[ch] = w[ch]
    return out

rows = []
for label, groups in SCENARIOS.items():
    merged = build_merged(w, groups, ACTIVE)
    corr = merged.corr()
    pairs = corr.where(np.triu(np.ones(corr.shape), 1).astype(bool)).stack()
    rows.append({"scenario": label, "n_channels": merged.shape[1],
                 "max |corr|": round(pairs.abs().max(), 2),
                 "max VIF": vif_table(merged).max()})
pd.DataFrame(rows).set_index("scenario")

In [ ]:
CHOSEN = "C: only fold meta_other into facebook"      # change after reviewing the table above
merged = build_merged(w, SCENARIOS[CHOSEN], ACTIVE)
print("channels:", list(merged.columns))
heatmap(merged.corr(), f"Raw correlation: {CHOSEN}")
vif_table(merged).to_frame("VIF")

## Decisions (fill in after reading the outputs)

- Channels dropped as too sparse:
- Channels merged and why:
- Channels kept separate and why:
- Final channel list for the model: